In [ ]:
%%spark

from pyspark.sql import functions as F

# Camada de explicabilidade da V2.
# A tabela de origem deve conter as pontua??es j? calculadas pelo fluxo principal.
tabela_origem = "sbx_t2i2016.v1_ana_edu_fin_cli"
df = spark.table(tabela_origem)

# ============================================================
# 1. PERCENTUAIS DERIVADOS DE SA?DA
# ============================================================
sem_saida_total = F.coalesce(F.col("VL_SAI_TOTAL"), F.lit(0)) == 0
den_saida = F.when(sem_saida_total, F.lit(None)).otherwise(F.col("VL_SAI_TOTAL"))

def pc_saida(coluna_valor):
    return F.when(
        sem_saida_total,
        F.lit(0.0)
    ).otherwise(
        F.round(F.coalesce(F.col(coluna_valor), F.lit(0.0)) / den_saida * 100, 2)
    )

df = (
    df
    .withColumn("PC_SAI_GEN",  pc_saida("VL_SAI_GEN"))
    .withColumn("PC_SAI_ESS",  pc_saida("VL_SAI_ESS"))
    .withColumn("PC_SAI_FLEX", pc_saida("VL_SAI_FLEX"))
    .withColumn("PC_SAI_RES",  pc_saida("VL_SAI_RES"))
    .withColumn("PC_SAI_CRED", pc_saida("VL_SAI_DIV"))
)

# ============================================================
# 2. TEXTOS DAS REGRAS DE CONCENTRA??O
# ============================================================
sem_saida_txt = "sem sa?da total no per?odo; pontua??o de concentra??o igual a 0"

df = (
    df
    .withColumn(
        "TX_REGRA_CONC_GEN",
        F.when(sem_saida_total, sem_saida_txt)
         .when(F.col("NR_PONT_CONC_GEN") == 0, "ficou menor ou igual ? refer?ncia de 75,00%")
         .when(F.col("NR_PONT_CONC_GEN") == 99, "ficou acima da refer?ncia de 75,00%")
         .otherwise("regra de categoriza??o n?o identificada")
    )
    .withColumn(
        "TX_REGRA_CONC_ESS",
        F.when(sem_saida_total, sem_saida_txt)
         .when(F.col("NR_PONT_CONC_ESS") == 0, "ficou abaixo de 50,00%")
         .when(F.col("NR_PONT_CONC_ESS") == 1, "ficou maior ou igual a 50,00% e menor que 75,00%")
         .when(F.col("NR_PONT_CONC_ESS") == 2, "ficou maior ou igual a 75,00%")
         .otherwise("regra de essenciais n?o identificada")
    )
    .withColumn(
        "TX_REGRA_CONC_FLEX",
        F.when(sem_saida_total, sem_saida_txt)
         .when(F.col("NR_PONT_CONC_FLEX") == 0, "ficou abaixo de 30,00%")
         .when(F.col("NR_PONT_CONC_FLEX") == 1, "ficou maior ou igual a 30,00% e menor que 45,00%")
         .when(F.col("NR_PONT_CONC_FLEX") == 2, "ficou maior ou igual a 45,00%")
         .otherwise("regra de flex?veis n?o identificada")
    )
    .withColumn(
        "TX_REGRA_CONC_RES",
        F.when(sem_saida_total, sem_saida_txt)
         .when(F.col("NR_PONT_CONC_RES") == 0, "ficou maior ou igual a 30,00%")
         .when(F.col("NR_PONT_CONC_RES") == 1, "ficou maior ou igual a 20,00% e menor que 30,00%")
         .when(F.col("NR_PONT_CONC_RES") == 2, "ficou abaixo de 20,00%")
         .otherwise("regra de reserva n?o identificada")
    )
    .withColumn(
        "TX_REGRA_CONC_CRED",
        F.when(sem_saida_total, sem_saida_txt)
         .when(F.col("NR_PONT_CONC_CRED") == 0, "ficou abaixo de 30,00%")
         .when(F.col("NR_PONT_CONC_CRED") == 1, "ficou maior ou igual a 30,00% e menor que 45,00%")
         .when(F.col("NR_PONT_CONC_CRED") == 2, "ficou maior ou igual a 45,00%")
         .otherwise("regra de cr?dito n?o identificada")
    )
)

# ============================================================
# 3. TEXTOS DO RESULTADO OR?AMENT?RIO
# ============================================================
sem_entrada_total = F.coalesce(F.col("VL_ENT_TOTAL"), F.lit(0)) == 0
tem_saida_total = F.coalesce(F.col("VL_SAI_TOTAL"), F.lit(0)) > 0

pc_sai_ent_pct = F.coalesce(F.col("PC_SAI_ENT"), F.lit(0.0)) * 100

df = df.withColumn(
    "TX_REGRA_RESULTADO",
    F.when(sem_entrada_total & ~tem_saida_total, "sem entradas e sem sa?das no per?odo")
     .when(sem_entrada_total & tem_saida_total, "sem entrada registrada e com sa?da no per?odo")
     .when(F.col("PC_SAI_ENT") < 0.75,  "comprometimento abaixo de 75,00%")
     .when(F.col("PC_SAI_ENT") < 0.95,  "comprometimento de 75,00% a 95,00%")
     .when(F.col("PC_SAI_ENT") < 1.00,  "comprometimento de 95,00% a 100,00%")
     .when(F.col("PC_SAI_ENT") <= 1.05, "comprometimento de 100,00% a 105,00%")
     .when(F.col("PC_SAI_ENT") <= 1.25, "comprometimento de 105,00% a 125,00%")
     .otherwise("comprometimento acima de 125,00%")
)

df = df.withColumn(
    "TX_REGRA_ORC_V2",
    F.when(
        F.col("TX_RES_ORC") == "Deficit?rio",
        "resultado deficit?rio: refor?a Gest?o do Or?amento, Consumo Planejado e Uso Consciente do Cr?dito"
    ).when(
        F.col("TX_RES_ORC") == "Superavit?rio",
        "resultado superavit?rio: refor?a Forma??o de Reserva"
    ).when(
        F.col("TX_RES_ORC") == "Neutro",
        "resultado neutro: n?o acrescenta pontua??o por or?amento"
    ).otherwise("resultado or?ament?rio n?o classificado")
)

# ============================================================
# 4. TEXTO DA REGRA DE PERFIL
# ============================================================
df = df.withColumn(
    "TX_REGRA_PERFIL_V2",
    F.when(
        F.col("NR_PONT_PRFL_ESS") == 1,
        "perfil financeiro refor?a Gest?o do Or?amento"
    ).when(
        F.col("NR_PONT_PRFL_FLEX") == 1,
        "perfil financeiro refor?a Consumo Planejado"
    ).when(
        F.col("NR_PONT_PRFL_RES") == 1,
        "perfil financeiro refor?a Forma??o de Reserva"
    ).when(
        F.col("NR_PONT_PRFL_CRED") == 1,
        "perfil financeiro refor?a Uso Consciente do Cr?dito"
    ).when(
        F.col("NM_MAC_PRFL_CLI") == "Investidor",
        "perfil Investidor n?o acrescenta pontua??o autom?tica na V2"
    ).otherwise(
        "perfil financeiro n?o acrescenta pontua??o na V2"
    )
)

# ============================================================
# 5. RESULTADO DOS TEMAS SEM DESEMPATE
# ============================================================
pontuacoes_finais = [
    "NR_PONT_CATEG",
    "NR_PONT_ORC",
    "NR_PONT_CONS",
    "NR_PONT_RES",
    "NR_PONT_CRED",
]

df = df.withColumn(
    "NR_MAIOR_PONTUACAO",
    F.greatest(*[F.coalesce(F.col(c), F.lit(0)) for c in pontuacoes_finais])
)

df = df.withColumn(
    "ARR_TEMAS_MAIOR_PONTUACAO",
    F.expr("""
        CASE
            WHEN NR_MAIOR_PONTUACAO > 0 THEN filter(array(
                CASE WHEN NR_PONT_CATEG = NR_MAIOR_PONTUACAO THEN 'Categoriza??o de gastos' END,
                CASE WHEN NR_PONT_ORC   = NR_MAIOR_PONTUACAO THEN 'Gest?o do or?amento' END,
                CASE WHEN NR_PONT_CONS  = NR_MAIOR_PONTUACAO THEN 'Consumo planejado' END,
                CASE WHEN NR_PONT_RES   = NR_MAIOR_PONTUACAO THEN 'Forma??o de reserva' END,
                CASE WHEN NR_PONT_CRED  = NR_MAIOR_PONTUACAO THEN 'Uso consciente do cr?dito' END
            ), x -> x is not null)
            ELSE cast(array() as array<string>)
        END
    """)
)

df = df.withColumn("QT_TEMAS_MAIOR_PONTUACAO", F.size("ARR_TEMAS_MAIOR_PONTUACAO"))
df = df.withColumn("TX_TEMAS_MAIOR_PONTUACAO", F.concat_ws(", ", "ARR_TEMAS_MAIOR_PONTUACAO"))

df = df.withColumn(
    "TX_RESULTADO_ASPECTO_DESTAQUE",
    F.when(
        F.col("NR_MAIOR_PONTUACAO") == 0,
        "nenhum tema apresentou pontua??o final maior que zero no per?odo"
    ).when(
        F.col("QT_TEMAS_MAIOR_PONTUACAO") > 1,
        F.concat_ws(
            "",
            F.lit("empate entre "),
            F.col("TX_TEMAS_MAIOR_PONTUACAO"),
            F.lit(", com "),
            F.col("NR_MAIOR_PONTUACAO").cast("string"),
            F.lit(" ponto(s). Nenhum desempate foi aplicado")
        )
    ).otherwise(
        F.concat_ws(
            "",
            F.lit("maior pontua??o em "),
            F.element_at("ARR_TEMAS_MAIOR_PONTUACAO", 1),
            F.lit(", com "),
            F.col("NR_MAIOR_PONTUACAO").cast("string"),
            F.lit(" ponto(s)")
        )
    )
)

# ============================================================
# 6. EXPLICA??O DAS F?RMULAS FINAIS DA V2
# ============================================================
def texto_final_tema(col_conc, col_orc, col_prfl, col_final):
    return F.when(
        F.coalesce(F.col(col_conc), F.lit(0)) == 0,
        F.concat_ws(
            "",
            F.lit("concentra??o 0; por regra V2, a pontua??o final ficou 0, mesmo com or?amento "),
            F.coalesce(F.col(col_orc), F.lit(0)).cast("string"),
            F.lit(" e perfil "),
            F.coalesce(F.col(col_prfl), F.lit(0)).cast("string")
        )
    ).otherwise(
        F.concat_ws(
            "",
            F.lit("concentra??o "),
            F.coalesce(F.col(col_conc), F.lit(0)).cast("string"),
            F.lit(" + or?amento "),
            F.coalesce(F.col(col_orc), F.lit(0)).cast("string"),
            F.lit(" + perfil "),
            F.coalesce(F.col(col_prfl), F.lit(0)).cast("string"),
            F.lit(" = "),
            F.coalesce(F.col(col_final), F.lit(0)).cast("string")
        )
    )

df = (
    df
    .withColumn("TX_FORMULA_FINAL_ORC", texto_final_tema("NR_PONT_CONC_ESS", "NR_PONT_ORC_ESS", "NR_PONT_PRFL_ESS", "NR_PONT_ORC"))
    .withColumn("TX_FORMULA_FINAL_CONS", texto_final_tema("NR_PONT_CONC_FLEX", "NR_PONT_ORC_FLEX", "NR_PONT_PRFL_FLEX", "NR_PONT_CONS"))
    .withColumn("TX_FORMULA_FINAL_RES", texto_final_tema("NR_PONT_CONC_RES", "NR_PONT_ORC_RES", "NR_PONT_PRFL_RES", "NR_PONT_RES"))
    .withColumn("TX_FORMULA_FINAL_CRED", texto_final_tema("NR_PONT_CONC_CRED", "NR_PONT_ORC_CRED", "NR_PONT_PRFL_CRED", "NR_PONT_CRED"))
)

# ============================================================
# 7. CLASSIFICA??O CONSOLIDADA
# ============================================================
df = df.withColumn(
    "NR_PONT_PERSONA",
    F.coalesce(F.col("NR_PONT_CONC_ESS"), F.lit(0))
    + F.coalesce(F.col("NR_PONT_CONC_FLEX"), F.lit(0))
    + F.coalesce(F.col("NR_PONT_CONC_RES"), F.lit(0))
    + F.coalesce(F.col("NR_PONT_CONC_CRED"), F.lit(0))
)

df = df.withColumn(
    "TX_FAIXA_CLASSIFICACAO_GERAL",
    F.when(F.col("NR_PONT_PERSONA").between(0, 2), "de 0 a 2 pontos")
     .when(F.col("NR_PONT_PERSONA").between(3, 4), "de 3 a 4 pontos")
     .when(F.col("NR_PONT_PERSONA").between(5, 6), "de 5 a 6 pontos")
     .when(F.col("NR_PONT_PERSONA").between(7, 8), "de 7 a 8 pontos")
     .otherwise("fora das faixas previstas")
)

df = df.withColumn(
    "TX_CLASSIFICACAO_GERAL",
    F.when(F.col("NR_PONT_PERSONA").between(0, 2), "Base Financeira Consolidada")
     .when(F.col("NR_PONT_PERSONA").between(3, 4), "Base Financeira Equilibrada")
     .when(F.col("NR_PONT_PERSONA").between(5, 6), "Oportunidade de Evolu??o Financeira")
     .when(F.col("NR_PONT_PERSONA").between(7, 8), "Prioridade para Organiza??o Financeira")
     .otherwise("Classifica??o n?o definida")
)

# ============================================================
# 8. CARACTER?STICAS DO PER?ODO
# ============================================================
df = (
    df
    .withColumn(
        "TX_CLASSIFICACAO_GEN",
        F.when(sem_saida_total, "sem sa?das no per?odo para classificar categoriza??o")
         .when(F.col("NR_PONT_CONC_GEN") == 0, "alta categoriza??o dos gastos")
         .when(F.col("NR_PONT_CONC_GEN") == 99, "baixa categoriza??o dos gastos")
         .otherwise("categoriza??o intermedi?ria dos gastos")
    )
    .withColumn(
        "TX_CLASSIFICACAO_ESS",
        F.when(sem_saida_total, "sem sa?das no per?odo para classificar despesas essenciais")
         .when(F.col("NR_PONT_CONC_ESS") == 0, "baixa concentra??o de despesas essenciais")
         .when(F.col("NR_PONT_CONC_ESS") == 1, "m?dia concentra??o de despesas essenciais")
         .when(F.col("NR_PONT_CONC_ESS") == 2, "alta concentra??o de despesas essenciais")
         .otherwise("concentra??o de despesas essenciais n?o classificada")
    )
    .withColumn(
        "TX_CLASSIFICACAO_FLEX",
        F.when(sem_saida_total, "sem sa?das no per?odo para classificar despesas flex?veis")
         .when(F.col("NR_PONT_CONC_FLEX") == 0, "baixa concentra??o de despesas flex?veis")
         .when(F.col("NR_PONT_CONC_FLEX") == 1, "m?dia concentra??o de despesas flex?veis")
         .when(F.col("NR_PONT_CONC_FLEX") == 2, "alta concentra??o de despesas flex?veis")
         .otherwise("concentra??o de despesas flex?veis n?o classificada")
    )
    .withColumn(
        "TX_CLASSIFICACAO_RES",
        F.when(sem_saida_total, "sem sa?das no per?odo para classificar forma??o de reserva")
         .when(F.col("NR_PONT_CONC_RES") == 0, "alta forma??o de reserva financeira")
         .when(F.col("NR_PONT_CONC_RES") == 1, "m?dia forma??o de reserva financeira")
         .when(F.col("NR_PONT_CONC_RES") == 2, "baixa forma??o de reserva financeira")
         .otherwise("forma??o de reserva n?o classificada")
    )
    .withColumn(
        "TX_CLASSIFICACAO_CRED",
        F.when(sem_saida_total, "sem sa?das no per?odo para classificar uso de cr?dito")
         .when(F.col("NR_PONT_CONC_CRED") == 0, "baixa utiliza??o de cr?dito")
         .when(F.col("NR_PONT_CONC_CRED") == 1, "m?dia utiliza??o de cr?dito")
         .when(F.col("NR_PONT_CONC_CRED") == 2, "alta utiliza??o de cr?dito")
         .otherwise("utiliza??o de cr?dito n?o classificada")
    )
)

# ============================================================
# 9. HELPERS DE FORMATA??O
# ============================================================
def brl(c):
    return F.translate(
        F.format_number(F.coalesce(F.col(c).cast("double"), F.lit(0.0)), 2),
        ",.",
        ".,"
    )

def pct(c):
    return F.translate(
        F.format_number(F.coalesce(F.col(c).cast("double"), F.lit(0.0)), 2),
        ",.",
        ".,"
    )

def pref(c):
    return F.translate(
        F.format_number(F.coalesce(F.col(c).cast("double"), F.lit(0.0)) * 100, 2),
        ",.",
        ".,"
    )

def s(c):
    return F.coalesce(F.col(c).cast("string"), F.lit("n?o informado"))

def n(c):
    return F.coalesce(F.col(c).cast("string"), F.lit("0"))

pc_sai_ent_fmt = F.when(
    sem_entrada_total & tem_saida_total,
    F.lit("sem denominador v?lido")
).otherwise(
    F.concat_ws(
        "",
        F.translate(F.format_number(pc_sai_ent_pct.cast("double"), 2), ",.", ".,"),
        F.lit("%")
    )
)

# ============================================================
# 10. TX_PERSONA_UNICA
# ============================================================
df = df.withColumn(
    "TX_PERSONA_UNICA",
    F.concat_ws(
        "",
        F.lit("LEITURA ?NICA POR CLIENTE\n\n"),
        F.lit("Cliente "), s("CD_CLI"), F.lit(" ? macroperfil "), s("NM_MAC_PRFL_CLI"),
        F.lit(", perfil de renda "), s("NM_PRFL"), F.lit(" e perfil financeiro "),
        s("NM_PRFL_FIN"), F.lit(".\n"),
        F.lit("Per?odo: "), s("DT_REF_INI"), F.lit(" a "), s("DT_REF_FIM"), F.lit(".\n\n"),

        F.lit("MOVIMENTA??O\n"),
        F.lit("- Entradas: R$ "), brl("VL_ENT_TOTAL"), F.lit(".\n"),
        F.lit("- Sa?das: R$ "), brl("VL_SAI_TOTAL"), F.lit(".\n"),
        F.lit("- Resultado: R$ "), brl("VL_ENT_TOTAL"), F.lit(" - R$ "), brl("VL_SAI_TOTAL"),
        F.lit(" = R$ "), brl("VL_RES_ORC"), F.lit(".\n"),
        F.lit("- Comprometimento: R$ "), brl("VL_SAI_TOTAL"), F.lit(" / R$ "), brl("VL_ENT_TOTAL"),
        F.lit(" = "), pc_sai_ent_fmt, F.lit(" ("), s("TX_REGRA_RESULTADO"),
        F.lit("; "), s("TX_STS_FINAL"), F.lit(").\n\n"),

        F.lit("CONCENTRA??O DOS GASTOS\n"),
        F.lit("- Categoriza??o: R$ "), brl("VL_SAI_GEN"), F.lit(" / R$ "), brl("VL_SAI_TOTAL"),
        F.lit(" = "), pct("PC_SAI_GEN"), F.lit("% ("), s("TX_REGRA_CONC_GEN"),
        F.lit("; refer?ncia: "), pref("PC_REF_GEN"), F.lit("%; pontua??o: "), n("NR_PONT_CONC_GEN"), F.lit(").\n"),
        F.lit("- Essenciais: R$ "), brl("VL_SAI_ESS"), F.lit(" / R$ "), brl("VL_SAI_TOTAL"),
        F.lit(" = "), pct("PC_SAI_ESS"), F.lit("% ("), s("TX_REGRA_CONC_ESS"),
        F.lit("; refer?ncia: "), pref("PC_REF_ESS"), F.lit("%; pontua??o: "), n("NR_PONT_CONC_ESS"), F.lit(").\n"),
        F.lit("- Flex?veis: R$ "), brl("VL_SAI_FLEX"), F.lit(" / R$ "), brl("VL_SAI_TOTAL"),
        F.lit(" = "), pct("PC_SAI_FLEX"), F.lit("% ("), s("TX_REGRA_CONC_FLEX"),
        F.lit("; refer?ncia: "), pref("PC_REF_FLEX"), F.lit("%; pontua??o: "), n("NR_PONT_CONC_FLEX"), F.lit(").\n"),
        F.lit("- Reserva: R$ "), brl("VL_SAI_RES"), F.lit(" / R$ "), brl("VL_SAI_TOTAL"),
        F.lit(" = "), pct("PC_SAI_RES"), F.lit("% ("), s("TX_REGRA_CONC_RES"),
        F.lit("; refer?ncia: "), pref("PC_REF_RES"), F.lit("%; pontua??o: "), n("NR_PONT_CONC_RES"), F.lit(").\n"),
        F.lit("- Cr?dito: R$ "), brl("VL_SAI_DIV"), F.lit(" / R$ "), brl("VL_SAI_TOTAL"),
        F.lit(" = "), pct("PC_SAI_CRED"), F.lit("% ("), s("TX_REGRA_CONC_CRED"),
        F.lit("; refer?ncia: "), pref("PC_REF_CRED"), F.lit("%; pontua??o: "), n("NR_PONT_CONC_CRED"), F.lit(").\n\n"),

        F.lit("PONTUA??O PELO OR?AMENTO\n"),
        F.lit("Base: "), s("TX_STS_FINAL"), F.lit(", com "), pc_sai_ent_fmt,
        F.lit(" de comprometimento. Regra V2: "), s("TX_REGRA_ORC_V2"), F.lit(".\n"),
        F.lit("- Categoriza??o: "), n("NR_PONT_ORC_GEN"), F.lit(".\n"),
        F.lit("- Essenciais: "), n("NR_PONT_ORC_ESS"), F.lit(".\n"),
        F.lit("- Flex?veis: "), n("NR_PONT_ORC_FLEX"), F.lit(".\n"),
        F.lit("- Reserva: "), n("NR_PONT_ORC_RES"), F.lit(".\n"),
        F.lit("- Cr?dito: "), n("NR_PONT_ORC_CRED"), F.lit(".\n\n"),

        F.lit("PONTUA??O PELO PERFIL\n"),
        F.lit("Base: "), s("NM_PRFL_FIN"), F.lit(". Regra V2: "), s("TX_REGRA_PERFIL_V2"), F.lit(".\n"),
        F.lit("- Categoriza??o: "), n("NR_PONT_PRFL_GEN"), F.lit(".\n"),
        F.lit("- Essenciais: "), n("NR_PONT_PRFL_ESS"), F.lit(".\n"),
        F.lit("- Flex?veis: "), n("NR_PONT_PRFL_FLEX"), F.lit(".\n"),
        F.lit("- Reserva: "), n("NR_PONT_PRFL_RES"), F.lit(".\n"),
        F.lit("- Cr?dito: "), n("NR_PONT_PRFL_CRED"), F.lit(".\n\n"),

        F.lit("RESULTADO POR TEMA\n"),
        F.lit("- Categoriza??o de gastos: "), n("NR_PONT_CATEG"),
        F.lit(" (igual ? pontua??o de concentra??o gen?rica).\n"),
        F.lit("- Gest?o do or?amento: "), n("NR_PONT_ORC"), F.lit(" ("), s("TX_FORMULA_FINAL_ORC"), F.lit(").\n"),
        F.lit("- Consumo planejado: "), n("NR_PONT_CONS"), F.lit(" ("), s("TX_FORMULA_FINAL_CONS"), F.lit(").\n"),
        F.lit("- Forma??o de reserva: "), n("NR_PONT_RES"), F.lit(" ("), s("TX_FORMULA_FINAL_RES"), F.lit(").\n"),
        F.lit("- Uso consciente do cr?dito: "), n("NR_PONT_CRED"), F.lit(" ("), s("TX_FORMULA_FINAL_CRED"), F.lit(").\n\n"),

        F.lit("ASPECTO FINANCEIRO DE MAIOR PONTUA??O\n"),
        F.lit("- "), s("TX_RESULTADO_ASPECTO_DESTAQUE"), F.lit(".\n\n"),

        F.lit("CLASSIFICA??O CONSOLIDADA\n"),
        F.lit("- C?lculo: "), n("NR_PONT_CONC_ESS"), F.lit(" + "), n("NR_PONT_CONC_FLEX"),
        F.lit(" + "), n("NR_PONT_CONC_RES"), F.lit(" + "), n("NR_PONT_CONC_CRED"),
        F.lit(" = "), n("NR_PONT_PERSONA"), F.lit(".\n"),
        F.lit("- Classifica??o: "), s("TX_CLASSIFICACAO_GERAL"),
        F.lit(" ("), s("TX_FAIXA_CLASSIFICACAO_GERAL"), F.lit(").\n\n"),

        F.lit("CARACTER?STICAS DO PER?ODO\n"),
        F.lit("- Categoriza??o: "), s("TX_CLASSIFICACAO_GEN"), F.lit(".\n"),
        F.lit("- Essenciais: "), s("TX_CLASSIFICACAO_ESS"), F.lit(".\n"),
        F.lit("- Flex?veis: "), s("TX_CLASSIFICACAO_FLEX"), F.lit(".\n"),
        F.lit("- Reserva: "), s("TX_CLASSIFICACAO_RES"), F.lit(".\n"),
        F.lit("- Cr?dito: "), s("TX_CLASSIFICACAO_CRED"), F.lit(".")
    )
)


In [ ]:
%%spark

from pyspark.sql import functions as F
from pyspark.sql.window import Window

amostra = (
    df
    .where(
        (F.col("FL_PARTICIPA_RADAR") == "S") &
        (F.col("TX_PERSONA_UNICA").isNotNull())
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("NM_MAC_PRFL_CLI")
                  .orderBy(F.rand())
        )
    )
    .where(F.col("rn") == 1)
    .select(
        "NM_MAC_PRFL_CLI",
        "CD_CLI",
        "TX_PERSONA_UNICA"
    )
    .orderBy("NM_MAC_PRFL_CLI")
    .collect()
)

for registro in amostra:
    print("\n" + "=" * 110)
    print(f"MACROPERFIL: {registro['NM_MAC_PRFL_CLI']}")
    print(f"CLIENTE: {registro['CD_CLI']}")
    print("=" * 110)

    texto = (
        registro["TX_PERSONA_UNICA"]
        .replace("&lt;br&gt;&lt;br&gt;", "\n\n")
        .replace("&lt;br&gt;", "\n")
    )

    print(texto)
